In [ ]:
# -*- coding: utf-8 -*-
"""
MIA–IRBI: Validação Empírica com Cinco Regimes Territoriais
Versão 2.1 — Cinco casos: SGO, Romagna, Zouping, Nova Nazaré, McDowell.
Pronto para execução no Google Colab.
"""

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from math import floor, degrees, atan2, sqrt

# ============================================================
# 1. DADOS DOS CINCO TERRITÓRIOS
# ============================================================
dados = pd.DataFrame({
    'municipio': [
        'São Gabriel do Oeste (BR)',
        'Romagna (IT)',
        'Zouping (CN)',
        'Nova Nazaré (BR)',
        'McDowell (EUA)'
    ],
    'E': [0.62, 0.84, 0.85, 0.32, 0.28],
    'X': [0.75, 0.82, 0.80, 0.28, 0.15],
    'M': [0.65, 0.73, 0.70, 0.08, 0.10],
    'PH_x': [0.50, 0.32, 0.40, 0.90, 0.95],
    'PH_y': [0.55, 0.60, 0.65, 0.92, 0.98],
    'Ph_x': [0.68, 0.80, 0.78, 0.12, 0.05],
    'Ph_y': [0.62, 0.70, 0.68, 0.10, 0.05]
})

# ============================================================
# 2. FUNÇÕES DO PROTOCOLO (VERSÃO 2.1)
# ============================================================
def calc_IRBI(E, X, M, deltas=(0.40, 0.35, 0.25)):
    """Índice de Resiliência Baseado em Ifá."""
    return deltas[0]*E + deltas[1]*X + deltas[2]*M

def vetor_R(row):
    """Vetor resultante: pressão exógena menos potência endógena."""
    return np.array([row['PH_x'] - row['Ph_x'], row['PH_y'] - row['Ph_y']])

def azimute(R):
    """Ângulo em graus: 0° = Leste, crescimento anti-horário."""
    ang = degrees(atan2(R[1], R[0]))
    return ang + 360 if ang < 0 else ang

def odu_territorial(ang):
    """Discretização em 256 odús (resolução de 1,40625°)."""
    return floor(ang / (360/256))

def proxy_captura_entropia(row):
    """
    Theta mediado = Theta_bruto * (1 - X).
    S_m = entropia mediada.
    S_potencial = S_m * X.
    S_dispersiva = S_m * (1 - X).
    """
    PH = np.array([row['PH_x'], row['PH_y']])
    Ph = np.array([row['Ph_x'], row['Ph_y']])
    mag_H = sqrt(PH[0]**2 + PH[1]**2)
    mag_h = sqrt(Ph[0]**2 + Ph[1]**2)
    denom = mag_H + mag_h
    Theta_bruto = mag_H / denom if denom != 0 else 0.5
    X = row['X']
    Theta = Theta_bruto * (1 - X)
    S_m = 1 - Theta
    S_potencial = S_m * X
    S_dispersiva = S_m * (1 - X)
    return Theta, S_m, S_potencial, S_dispersiva

def classificar_regime(Theta, S_m, M, R_x, R_y, E, X):
    """
    Classificação nos cinco regimes territoriais.
    Colapso Exofágico: Theta >= 0.75, M < 0.3, X < 0.4, E < 0.5.
    Emergência Sintrópica: R_x < 0 e R_y < 0.
    """
    if Theta >= 0.75 and M < 0.3 and X < 0.4 and E < 0.5:
        return "Colapso exofágico"
    if S_m < 0.35 and M < 0.4:
        return "Homeostase rígida"
    if S_m > 0.7 and M < 0.4:
        return "Entropia dispersiva"
    if S_m > 0.5 and M > 0.5:
        if R_x < 0 and R_y < 0 and (E > 0.5 or X > 0.5):
            return "Emergência sintrópica"
        else:
            return "Entropia potencial" if X >= 0.5 else "Entropia dispersiva"
    elif S_m > 0.5 and M <= 0.5:
        return "Entropia dispersiva"
    else:
        if R_x < 0 and R_y < 0:
            return "Emergência sintrópica"
        return "Entropia potencial"

# ============================================================
# 3. PIPELINE DE DIAGNÓSTICO
# ============================================================
df = dados.copy()

df['IRBI'] = df.apply(lambda r: calc_IRBI(r['E'], r['X'], r['M']), axis=1)
df['R_vetor'] = df.apply(vetor_R, axis=1)
df['R_x'] = df['R_vetor'].apply(lambda v: v[0])
df['R_y'] = df['R_vetor'].apply(lambda v: v[1])
df['R_norm'] = df['R_vetor'].apply(lambda v: sqrt(v[0]**2 + v[1]**2))
df['Angulo'] = df['R_vetor'].apply(azimute)
df['Odu'] = df['Angulo'].apply(odu_territorial)

entropias = df.apply(lambda r: proxy_captura_entropia(r), axis=1, result_type='expand')
entropias.columns = ['Theta', 'S_m', 'S_potencial', 'S_dispersiva']
df = pd.concat([df, entropias], axis=1)

df['Regime'] = df.apply(lambda r: classificar_regime(
    r['Theta'], r['S_m'], r['M'], r['R_x'], r['R_y'], r['E'], r['X']), axis=1)

# ============================================================
# 4. TABELA RESUMO
# ============================================================
colunas_resumo = [
    'municipio', 'IRBI', 'Theta', 'S_m', 'S_potencial',
    'S_dispersiva', 'Angulo', 'Odu', 'Regime'
]
print("=== DIAGNÓSTICO MIA-IRBI v2.1 — CINCO REGIMES ===")
display(df[colunas_resumo].round(3))

# ============================================================
# 5. GRÁFICO POLAR (S_m como raio, ângulo = direção de R)
# ============================================================
cores = {
    "Emergência sintrópica": "#1f77b4",
    "Homeostase rígida": "#d62728",
    "Entropia dispersiva": "#ff7f0e",
    "Entropia potencial": "#2ca02c",
    "Colapso exofágico": "#8c0032"
}

fig1 = go.Figure()
for regime, cor in cores.items():
    subset = df[df['Regime'] == regime]
    if subset.empty:
        continue
    fig1.add_trace(go.Scatterpolar(
        r=subset['S_m'],
        theta=subset['Angulo'],
        mode='markers+text',
        name=regime,
        marker=dict(size=16, color=cor, line=dict(width=1.5, color='black')),
        text=subset['municipio'].str.replace(' (BR)', '').str.replace(' (IT)', '').str.replace(' (CN)', '').str.replace(' (EUA)', ''),
        textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'IRBI: %{customdata[0]:.3f}<br>'
            'Θ: %{customdata[1]:.3f}<br>'
            'S_m: %{r:.3f}<br>'
            'S_pot: %{customdata[2]:.3f}<br>'
            'S_disp: %{customdata[3]:.3f}<br>'
            'Ângulo: %{theta:.1f}°<br>'
            'Odú: %{customdata[4]}'
        ),
        customdata=subset[['IRBI', 'Theta', 'S_potencial', 'S_dispersiva', 'Odu']].values
    ))

fig1.update_layout(
    title=dict(text="Círculo Azimutal MIA–IRBI — Cinco Regimes Territoriais", x=0.5, font=dict(size=18)),
    polar=dict(
        radialaxis=dict(range=[0, 1.05], dtick=0.2, showline=True, linewidth=1, gridcolor='lightgrey', title="S_m"),
        angularaxis=dict(direction='counterclockwise', rotation=0, dtick=45, gridcolor='lightgrey')
    ),
    showlegend=True,
    legend=dict(x=0.85, y=0.1, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=20, r=20, t=60, b=20)
)
fig1.show()

# ============================================================
# 6. GRÁFICO S_potencial × S_dispersiva
# ============================================================
fig2 = go.Figure()
for regime, cor in cores.items():
    subset = df[df['Regime'] == regime]
    if subset.empty:
        continue
    fig2.add_trace(go.Scatter(
        x=subset['S_dispersiva'],
        y=subset['S_potencial'],
        mode='markers+text',
        name=regime,
        marker=dict(size=16, color=cor, line=dict(width=1.5, color='black')),
        text=subset['municipio'].str.replace(' (BR)', '').str.replace(' (IT)', '').str.replace(' (CN)', '').str.replace(' (EUA)', ''),
        textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'S_pot: %{y:.3f}<br>'
            'S_disp: %{x:.3f}<br>'
            'IRBI: %{customdata[0]:.3f}<br>'
            'Θ: %{customdata[1]:.3f}<br>'
            'Regime: %{customdata[2]}'
        ),
        customdata=subset[['IRBI', 'Theta', 'Regime']].values
    ))

max_val = max(df['S_potencial'].max(), df['S_dispersiva'].max()) * 1.1
fig2.add_trace(go.Scatter(
    x=[0, max_val],
    y=[0, max_val],
    mode='lines',
    name='S_pot = S_disp',
    line=dict(dash='dash', color='grey'),
    showlegend=True
))

fig2.update_layout(
    title=dict(text="Entropia Potencial vs Dispersiva — Cinco Regimes", x=0.5, font=dict(size=18)),
    xaxis=dict(title="S_dispersiva (fragmentação)", range=[0, max_val], dtick=0.1, showgrid=True),
    yaxis=dict(title="S_potencial (reorganização)", range=[0, max_val], dtick=0.1, showgrid=True),
    showlegend=True,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=20, r=20, t=60, b=20),
    width=700,
    height=600
)
fig2.show()

# ============================================================
# 7. EXPORTAÇÃO
# ============================================================
df[colunas_resumo].to_csv('validacao_MIA_IRBI_cinco_regimes.csv', index=False)
df[colunas_resumo].to_excel('validacao_MIA_IRBI_cinco_regimes.xlsx', index=False)
print("Resultados salvos em:")
print("- validacao_MIA_IRBI_cinco_regimes.csv")
print("- validacao_MIA_IRBI_cinco_regimes.xlsx")
